# **과제 3. 성능 개선 실험**



In [3]:
!pip install transformers datasets accelerate

# [1. 데이터 증강 실험]

In [4]:
import pandas as pd
from datasets import Dataset
from transformers import PreTrainedTokenizerFast, BartForConditionalGeneration
from sklearn.model_selection import train_test_split

# 1. 모델과 토크나이저 불러오기 (KoBART)
model_name = "gogamza/kobart-base-v2"
tokenizer = PreTrainedTokenizerFast.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

# 2. 증강된 데이터(22,000개) 불러오기
print("증강 데이터 불러오는 중...")
data = pd.read_csv("augmented_train_v2.csv")

# 학습용(80%)과 검증용(20%)으로 분리
train_df, val_df = train_test_split(data, test_size=0.2, random_state=42)
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

# 3. 토크나이징 (문장을 AI가 이해할 수 있는 숫자로 쪼개기)
def preprocess_function(examples):
    inputs = tokenizer(examples["input"], max_length=64, truncation=True, padding="max_length")
    targets = tokenizer(examples["output"], max_length=64, truncation=True, padding="max_length")
    inputs["labels"] = targets["input_ids"]
    return inputs

tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_val = val_dataset.map(preprocess_function, batched=True)

print("[데이터 증강 실험] 학습 데이터 세팅 완료!")

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

증강 데이터 불러오는 중...


Map:   0%|          | 0/18020 [00:00<?, ? examples/s]

Map:   0%|          | 0/4506 [00:00<?, ? examples/s]

✅ 데이터 세팅 완료!


In [5]:
import numpy as np
from transformers import BartForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer

# 1. 모델 불러오기
model = BartForConditionalGeneration.from_pretrained(model_name)

# 2. F1 점수 계산기 준비
def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    f1_scores = []
    for pred, true in zip(decoded_preds, decoded_labels):
        matches = sum(1 for a, b in zip(pred, true) if a == b)
        precision = matches / max(len(pred), 1)
        recall = matches / max(len(true), 1)
        if precision + recall == 0:
            f1_scores.append(0)
        else:
            f1_scores.append(2 * precision * recall / (precision + recall))
    return {"char_f1": sum(f1_scores) / len(f1_scores)}

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

# [2. 하이퍼파라미터 튜닝]

In [10]:
# 3. 🎯 하이퍼파라미터 튜닝 세팅 (과제 조건: 최소 3개 이상 변경)
training_args = Seq2SeqTrainingArguments(
    output_dir="./kobart_tuned_results",

    # --- 튜닝 변경 포인트 3가지 ---
    learning_rate=1e-5,             # [변경 포인트 1] Learning Rate
    per_device_train_batch_size=16, # [변경 포인트 2] Batch Size
    num_train_epochs=5,             # [변경 포인트 3] Epoch
    # ------------------------------

    per_device_eval_batch_size=16,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    predict_with_generate=True,
    fp16=True,
    load_best_model_at_end=True,
)

# 4. 트레이너 장착 및 진짜 학습 시작
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

print("세팅 완료. 본격적인 파인튜닝 학습을 시작합니다.")
trainer.train()

세팅 완료. 본격적인 파인튜닝 학습을 시작합니다.


Epoch,Training Loss,Validation Loss,Char F1
1,0.489299,0.579950,0.500148
2,0.469546,0.526658,0.521595
3,0.432167,0.504844,0.535704
4,0.385094,0.493732,0.541866


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Char F1
1,0.489299,0.579950,0.500148
2,0.469546,0.526658,0.521595
3,0.432167,0.504844,0.535704
4,0.385094,0.493732,0.541866
5,0.364757,0.490571,0.544403


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=5635, training_loss=0.4370090880643613, metrics={'train_runtime': 1138.812, 'train_samples_per_second': 79.118, 'train_steps_per_second': 4.948, 'total_flos': 3433578430464000.0, 'train_loss': 0.4370090880643613, 'epoch': 5.0})

# [3. 최종 예측]

In [11]:
# 최종 예측 코드
print("테스트 데이터 예측을 시작합니다...")
test_df = pd.read_csv("test.csv")
test_dataset = Dataset.from_pandas(test_df)

def generate_predictions(batch):
    inputs = tokenizer(batch["input"], max_length=128, return_tensors="pt", truncation=True, padding=True).to('cuda')
    # 가장 높은 확률 문장을 생성하는 Beam Search 적용
    outputs = model.generate(inputs["input_ids"], max_length=128, num_beams=5)
    batch["pred"] = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    return batch

results = test_dataset.map(generate_predictions, batched=True, batch_size=16)

# 최종 제출 파일 저장
submission = pd.read_csv("sample_submission.csv")
submission["output"] = results["pred"]
submission.to_csv("submission_final.csv", index=False, encoding="utf-8-sig")

print("submission_final.csv' 파일을 다운로드")

테스트 데이터 예측을 시작합니다...


Parameter 'function'=<function generate_predictions at 0x7fabaee8c0e0> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Map:   0%|          | 0/1689 [00:00<?, ? examples/s]

submission_final.csv' 파일을 다운로드


## [4. 오류 분석]

In [13]:
pip install python-Levenshtein

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 74.5 MB/s eta 0:00:00


In [16]:
import numpy as np
import pandas as pd
from datasets import Dataset
from nltk.translate.bleu_score import sentence_bleu
import Levenshtein

# 1. 검증 데이터셋에서 정답과 예측값 추출
# train_test_split으로 나눴던 val_dataset을 직접 사용합니다.
val_dataset = Dataset.from_pandas(val_df) # 아까 만든 val_dataset 재사용

def get_preds_and_labels(batch):
    inputs = tokenizer(batch["input"], max_length=128, return_tensors="pt", truncation=True, padding=True).to('cuda')
    outputs = model.generate(inputs["input_ids"], max_length=128, num_beams=5)
    batch["pred"] = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    batch["true"] = batch["output"]
    return batch

# 예측 실행
val_with_preds = val_dataset.map(get_preds_and_labels, batched=True, batch_size=16)
val_results = pd.DataFrame({'pred': val_with_preds['pred'], 'true': val_with_preds['true']})

# 2. 분석 함수 정의
def analyze_errors(row):
    pred = row['pred']
    true = row['true']
    bleu = sentence_bleu([list(true)], list(pred))
    edit_dist = Levenshtein.distance(pred, true)
    return pd.Series([bleu, edit_dist])

# 3. 분석 수행
print("🔍 오류 분석을 다시 시작합니다...")
val_results[['bleu', 'edit_dist']] = val_results.apply(analyze_errors, axis=1)

# 4. 결과 출력
print("--- 분석 결과 요약 ---")
print(f"평균 BLEU Score: {val_results['bleu'].mean():.4f}")
print(f"평균 Edit Distance: {val_results['edit_dist'].mean():.4f}")
print("\n--- 가장 많이 틀린 사례 (Top 10) ---")
print(val_results.sort_values(by='edit_dist', ascending=False).head(10)[['true', 'pred', 'edit_dist']])

Map:   0%|          | 0/4506 [00:00<?, ? examples/s]

🔍 오류 분석을 다시 시작합니다...


/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_

--- 분석 결과 요약 ---
평균 BLEU Score: 0.6834
평균 Edit Distance: 30.5404

--- 가장 많이 틀린 사례 (Top 10) ---
                                                   true  \
3027  저녁 시간 직원은 나름 친절한데 아침 시간 직원들은 그냥 서비스 마인드가 1도 없는...   
2282  저녁 시간 직원은 나름 친절한데 아침 시간 직원들은 그냥 서비스 마인드가 1도 없는...   
3477  재벌 아니면 안 가는 걸 추천 서비스 마인드가 심각한 곳 1 유팡젖병 소독기을 체크...   
160   필자는 프리미엄룸을 이용함 지극히 제 개인적인 생각 및 느낀 걸로 적겠음 장점 1 ...   
3908  내 인생에 후기가 처음이다 진짜 제발 가지 마세요 들어가려면 매점에서 1만 원 이상...   
1917  이마트24 편의점이 1층에 있고 아이스크림 맛집 백미당이 1층에 있으며 2층에서 생...   
3199  2023년 7월경 용인특례시에서 온 청년으로서 처음 방문한 후기를 남깁니다 캠핑장이...   
3149  프리미어 클럽 더블 룸 예약 시설은 큰 불만 없음 살짝씩 조금 오래된 감은 있으나 ...   
3259  일단 굉장히 깔끔함 침대나 인테리어 이 정도면 매우 양호 4인실을 일행 4명이 사용...   
3655  댓글 믿지 마세요 사장님이 좋은 후기만 걸어두십니다 나도 후기 좋다고 믿고 갔다가 ...   

                                                   pred  edit_dist  
3027  저녁 시간 직원은 나름 친절한테 아침 식사 준비는 그냥 서비스가 마인드가 1도 없는...     1065.0  
2282  저녁 시간 직원은 나름 친절한데 아침 시간 직원들은 그냥 서비스 마인드가 1도 없는...      870.0  
3477  재벌 아니면 안 가는 걸 추천 서비스 마인드가 심각한 곳 1 유팡젖병 소독기